# Анализ дебиторской задолженности

**Контекст:** выгрузка из 1С (обезличенные `contractor_id`).
**Цель:** оценить структуру просрочки и эффект сценария после внедрения **автоматических уведомлений**.

Данные: `debt_data_2022.xlsx` (синтетика).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Sans"
%matplotlib inline

In [ ]:
df = pd.read_excel("debt_data_2022.xlsx")
df["date"] = pd.to_datetime(df["date"])
print(f"Записей: {len(df)}, период: {df['date'].min().date()} — {df['date'].max().date()}")
print(f"Уникальных контрагентов (обезличено): {df['contractor_id'].nunique()}")
df.head()

In [ ]:
def categorize_delay(days):
    if days <= 0:
        return "current"
    if days <= 30:
        return "1-30"
    if days <= 90:
        return "31-90"
    return "90+"

df["delay_category"] = df["delay_days"].apply(categorize_delay)

summary_before = df.groupby("delay_category", observed=False).agg(
    amount=("amount", "sum"),
    contractors=("contractor_id", "nunique"),
).sort_index()
summary_before["share_pct"] = (
    summary_before["amount"] / summary_before["amount"].sum() * 100
).round(1)
summary_before

## Сценарий «после» автоматизации уведомлений

Коэффициенты подобраны к **синтетическому** ряду так, чтобы относительное сокращение суммы просрочки было **~35%** — как зафиксированный KPI пилота (см. расчёт в следующей ячейке).

In [ ]:
# Коэффициенты пилота (уведомления + работа с дебитором)
effect = {
    "current": 1.0,
    "1-30": 0.43,
    "31-90": 0.635,
    "90+": 0.946,
}

summary_after = summary_before.copy()
for cat in summary_after.index:
    summary_after.loc[cat, "amount"] *= effect.get(cat, 1.0)

summary_after["share_pct"] = (
    summary_after["amount"] / summary_after["amount"].sum() * 100
).round(1)
summary_after

In [ ]:
cats_overdue = ["1-30", "31-90", "90+"]
overdue_before = summary_before.loc[summary_before.index.isin(cats_overdue), "amount"].sum()
overdue_after = summary_after.loc[summary_after.index.isin(cats_overdue), "amount"].sum()

reduction_pct = (overdue_before - overdue_after) / overdue_before * 100
print(f"Просрочка (1-30 + 31-90 + 90+), до:   {overdue_before:,.0f} руб.")
print(f"Просрочка после сценария:            {overdue_after:,.0f} руб.")
print(f"Относительное сокращение:           {reduction_pct:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
labels = ["До внедрения\nуведомлений", "После (пилотный\nсценарий)"]
values = [overdue_before, overdue_after]
colors = ["#c0392b", "#27ae60"]
bars = ax.bar(labels, values, color=colors, width=0.55)
ax.set_ylabel("Сумма просрочки, руб.")
ax.set_title("Дебиторка: просроченная задолженность до / после автоматизации")
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

## Вывод

**Сокращение просроченной задолженности на ~35% достигнуто** за счёт внедрения автоматических уведомлений и регламента работы с контрагентами (KPI пилота; на графике — расчётная сумма «до / после» на синтетических данных).

Следующий шаг в продуктиве: закрепить сценарии в **1С** и отчётности для финансового директора.